# Sasya AI - Baseline Training (Kaggle)
This notebook trains the baseline ResNet-50 model on the Kaggle PlantVillage dataset, filtered for Maize, Tomato, Grape, and Potato.


In [ ]:
import os
import shutil
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


In [ ]:
import os
import shutil

KAGGLE_INPUT_DIR = "/kaggle/input"
WORKING_DIR = "/kaggle/working/data"
TARGET_CROPS = ["Corn", "Tomato", "Grape", "Potato", "Maize"]

print("Finding dataset directories...")
train_src = None
valid_src = None
for root, dirs, files in os.walk(KAGGLE_INPUT_DIR):
    # Only pick leaf nodes that are actual dataset splits, avoiding hidden or extra dirs
    if os.path.basename(root) == "train" and not train_src:
        train_src = root
    if os.path.basename(root) == "valid" and not valid_src:
        valid_src = root

print(f"Found train dir: {train_src}")
print(f"Found valid dir: {valid_src}")

if train_src and valid_src:
    for split_name, src_split in [("train", train_src), ("valid", valid_src)]:
        dst_split = os.path.join(WORKING_DIR, split_name)
        os.makedirs(dst_split, exist_ok=True)
        for class_name in os.listdir(src_split):
            if any(crop in class_name for crop in TARGET_CROPS):
                src_class = os.path.join(src_split, class_name)
                dst_class = os.path.join(dst_split, class_name)
                if not os.path.exists(dst_class):
                    shutil.copytree(src_class, dst_class)
else:
    raise FileNotFoundError("Could not find train/valid directories in input")
print("Filtering complete. Data ready in /kaggle/working/data")



In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'valid': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

train_dir = os.path.join(WORKING_DIR, 'train')
val_dir = os.path.join(WORKING_DIR, 'valid')

image_datasets = {
    'train': datasets.ImageFolder(train_dir, data_transforms['train']),
    'valid': datasets.ImageFolder(val_dir, data_transforms['valid'])
}

batch_size = 32
dataloaders = {
    'train': DataLoader(image_datasets['train'], batch_size=batch_size, shuffle=True, num_workers=2),
    'valid': DataLoader(image_datasets['valid'], batch_size=batch_size, shuffle=False, num_workers=2)
}

class_names = image_datasets['train'].classes
num_classes = len(class_names)
print(f"Found {num_classes} classes: {class_names}")


In [ ]:
def build_model(num_classes):
    model = models.resnet50(weights="IMAGENET1K_V2")
    # Freeze all layers
    for param in model.parameters():
        param.requires_grad = False
    # Unfreeze layer4
    for param in model.layer4.parameters():
        param.requires_grad = True
    # Replace final FC layer
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = build_model(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.fc.parameters(), 'lr': 1e-4}
])


In [ ]:
num_epochs = 15

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    print("-" * 10)
    
    for phase in ['train', 'valid']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in dataloaders[phase]:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(image_datasets[phase])
        epoch_acc = running_corrects.double() / len(image_datasets[phase])

        print(f"{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

print("Training complete!")


In [ ]:
print("Running full evaluation on Validation set...")
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in dataloaders['valid']:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(15, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png')
plt.show()


In [ ]:
# Save the model
weights_path = "/kaggle/working/resnet50_sasya_baseline.pth"
torch.save(model.state_dict(), weights_path)
print(f"Model weights saved to {weights_path}")
print("You can download this file from the Kaggle Output panel on the right sidebar.")
